In [ ]:
import numpy as np
from scipy.spatial.distance import cdist
#def calcola_distanze(A, strategie_giocatore_2, lunghezza2):
#    D_strategica = np.zeros((lunghezza2, lunghezza2))
#    D_fisica = np.zeros((lunghezza2, lunghezza2))
#    for j in range(lunghezza2):
#        for k in range(lunghezza2):
#            D_strategica[j, k] = np.linalg.norm(np.sort(A[:, j]) - np.sort(A[:, k]))
#            D_fisica[j, k] = np.sum(np.abs(np.array(strategie_giocatore_2[j]) - np.array(strategie_giocatore_2[k]))) / 2.0
#
#    D_strat_norm = D_strategica / np.max(D_strategica)
#    D_fisica_norm = D_fisica / np.max(D_fisica)
#
#    return D_strat_norm, D_fisica_norm





def calcola_distanze(A, strategie_giocatore_2, lunghezza2):
    # 1. Pre-ordiniamo le colonne di A per avere i profili ordinati (lunghezza1 x lunghezza2)
    A_sort = np.sort(A, axis=0)
    
    # Trasponiamo A_sort (diventa lunghezza2 x lunghezza1) perché cdist si aspetta le osservazioni sulle righe
    A_sort_T = A_sort.T
    
    # 2. Calcolo Distanza Strategica (Euclidea) istantaneo tramite SciPy
    # Equivale matematicamente a: np.linalg.norm(A_sort[:, j] - A_sort[:, k])
    D_strategica = cdist(A_sort_T, A_sort_T, metric='euclidean')
    
    # 3. Trasformiamo le strategie in matrice
    S2 = np.array(strategie_giocatore_2)
    
    # 4. Calcolo Distanza Fisica (Manhattan / Cityblock) istantaneo tramite SciPy
    # Equivale matematicamente a: np.sum(np.abs(S2[j] - S2[k])) / 2.0
    D_fisica = cdist(S2, S2, metric='cityblock') / 2.0
    
    # 5. Normalizzazione originale intatta
    D_strat_norm = D_strategica / np.max(D_strategica)
    D_fisica_norm = D_fisica / np.max(D_fisica)
    
    return D_strat_norm, D_fisica_norm

In [2]:
import numpy as np

def probabilita_dirichlet(conteggi_reali, storico):
    p= conteggi_reali / len(storico)
    return p

In [ ]:
import numpy as np

def cross_validation_2d(
    storico,
    D_strat_norm,
    D_fisica_norm,
    griglia_theta,
    griglia_sigma,
    alpha,
    somma_alpha=None,
    min_round=5,
    return_diagnostics=False,
    tie_tol=1e-8
):
    """
    Seleziona theta e sigma con validazione PREQUENTIALE one-step-ahead.

    Per ogni t = min_round, ..., N-1:
      1) stima p usando SOLO storico[:t];
      2) valuta la probabilita' assegnata alla mossa storico[t];
      3) aggiunge la mossa osservata al training set.

    Vantaggi rispetto alla versione precedente:
      - funziona gia' con 10 round (se min_round < 10);
      - nessuna finestra futura sovrapposta;
      - nessun look-ahead / leakage temporale;
      - aggiornamento cumulativo O(N) per ogni coppia (theta, sigma),
        invece di risommare ogni prefisso da zero.

    Parametri
    ---------
    min_round : int
        Numero minimo di osservazioni usate prima della prima previsione.
        Default 5: con uno storico di 10 round produce 5 previsioni out-of-sample.
    return_diagnostics : bool
        Se True restituisce anche superficie degli score e indicatori di
        quanto theta/sigma siano ben separati sulla griglia.
    tie_tol : float
        Tolleranza numerica per considerare due score praticamente uguali.

    Returns
    -------
    theta_opt, sigma_opt
    oppure, con return_diagnostics=True:
    theta_opt, sigma_opt, diagnostica
    """
    storico = np.asarray(storico, dtype=int).ravel()
    alpha = np.asarray(alpha, dtype=float).ravel()
    D_strat_norm = np.asarray(D_strat_norm, dtype=float)
    D_fisica_norm = np.asarray(D_fisica_norm, dtype=float)
    griglia_theta = np.asarray(griglia_theta, dtype=float)
    griglia_sigma = np.asarray(griglia_sigma, dtype=float)

    numstrategie = len(alpha)
    N = len(storico)

    if somma_alpha is None:
        somma_alpha = float(np.sum(alpha))
    else:
        somma_alpha = float(somma_alpha)

    if N <= min_round:
        raise ValueError(
            f"Servono piu' di min_round={min_round} osservazioni; ricevute N={N}."
        )
    if min_round < 1:
        raise ValueError("min_round deve essere almeno 1.")
    if np.any(storico < 0) or np.any(storico >= numstrategie):
        raise ValueError("storico contiene indici di strategia non validi.")
    if D_strat_norm.shape != (numstrategie, numstrategie):
        raise ValueError("D_strat_norm ha dimensioni incompatibili con alpha.")
    if D_fisica_norm.shape != (numstrategie, numstrategie):
        raise ValueError("D_fisica_norm ha dimensioni incompatibili con alpha.")
    if np.any(griglia_sigma <= 0):
        raise ValueError("Tutti i valori di sigma devono essere > 0.")
    if len(griglia_theta) == 0 or len(griglia_sigma) == 0:
        raise ValueError("Le griglie di theta e sigma non possono essere vuote.")
    if np.any(alpha <= 0):
        raise ValueError("Per il log-score e' richiesto alpha > 0 per ogni strategia.")

    # Score medio log-predittivo per osservazione.
    # Usare la media (anziche' la somma) non cambia l'ottimo, ma rende
    # confrontabili diagnostiche ottenute con storici di lunghezze diverse.
    scores = np.full((len(griglia_theta), len(griglia_sigma)), -np.inf)
    tiny = np.finfo(float).tiny

    for i_theta, theta in enumerate(griglia_theta):
        D_ibr = (1.0 - theta) * D_strat_norm + theta * D_fisica_norm

        for i_sigma, sigma in enumerate(griglia_sigma):
            K = np.exp(-(D_ibr ** 2) / (2.0 * sigma ** 2))
            somme_riga = K.sum(axis=1, keepdims=True)
            K_norm = K / somme_riga

            # Stato del modello dopo i primi min_round esempi.
            spill = K_norm[storico[:min_round]].sum(axis=0)
            ll_tot = 0.0

            # Prequential one-step-ahead: ogni osservazione viene testata
            # prima di essere aggiunta al training set.
            for t in range(min_round, N):
                denom = float(np.sum(spill) + somma_alpha)
                prob = (spill + alpha) / denom
                y = storico[t]
                ll_tot += np.log(max(float(prob[y]), tiny))

                # Aggiornamento cumulativo O(numstrategie).
                spill += K_norm[y]

            scores[i_theta, i_sigma] = ll_tot / (N - min_round)

    # Ottimo sulla griglia. In caso di parita' numerica np.argmax sceglie
    # il primo; per questo restituiamo anche una diagnostica di ambiguita'.
    flat_best = int(np.argmax(scores))
    i_best, j_best = np.unravel_index(flat_best, scores.shape)
    theta_opt = float(griglia_theta[i_best])
    sigma_opt = float(griglia_sigma[j_best])
    best_score = float(scores[i_best, j_best])

    # Profili: per ciascun theta/sigma si prende il miglior valore dell'altro
    # iperparametro. Sono utili per capire se l'ottimo e' davvero informativo.
    theta_profile = scores.max(axis=1)
    sigma_profile = scores.max(axis=0)

    theta_order = np.sort(theta_profile)[::-1]
    sigma_order = np.sort(sigma_profile)[::-1]
    delta_theta = float(theta_order[0] - theta_order[1]) if len(theta_order) > 1 else np.inf
    delta_sigma = float(sigma_order[0] - sigma_order[1]) if len(sigma_order) > 1 else np.inf

    # Conta quante coppie sono numericamente indistinguibili dall'ottimo.
    near_best = np.argwhere(np.abs(scores - best_score) <= tie_tol)

    diagnostica = {
        "score_medio_log_predittivo": best_score,
        "n_train_iniziale": int(min_round),
        "n_predizioni_out_of_sample": int(N - min_round),
        "score_grid": scores,
        "theta_profile": theta_profile,
        "sigma_profile": sigma_profile,
        "delta_theta_score": delta_theta,
        "delta_sigma_score": delta_sigma,
        "n_coppie_in_parita_numerica": int(len(near_best)),
        "theta_ambiguo": bool(delta_theta <= tie_tol),
        "sigma_ambiguo": bool(delta_sigma <= tie_tol),
    }

    if return_diagnostics:
        return theta_opt, sigma_opt, diagnostica
    return theta_opt, sigma_opt


In [ ]:
import numpy as np

def aggrega_per_classi(prob_singole, nomi_classi, classi_giocatore_2, strategie_giocatore_2):
    prob_classi = np.zeros(len(nomi_classi))
    for i, nome_classe in enumerate(nomi_classi):
        for mossa in classi_giocatore_2[nome_classe]:
            indice = strategie_giocatore_2.index(mossa)
            prob_classi[i] += prob_singole[indice]
    return prob_classi

In [ ]:
def genera_strategie(soldati, zone):
    """
    Funzione ricorsiva per generare tutte le possibili distribuzioni 
    di 'soldati' in un numero definito di 'zone'.
    """
    # Caso base: se c'è solo una zona, tutti i soldati rimanenti vanno lì
    if zone == 1:
        yield (soldati,)
        return
    
    # Itera su tutti i possibili soldati da assegnare alla zona corrente
    # Partiamo dal numero massimo per mantenere un ordine simile al tuo codice originale
    for i in range(soldati, -1, -1):
        for resto in genera_strategie(soldati - i, zone - 1):
            yield (i,) + resto

In [ ]:
def raggruppa_in_classi(strategie):
    """
    Raggruppa le strategie in classi di equivalenza (es. "4-0-0").
    """
    classi = {}
    for s in strategie:
        # Ordina la tupla in ordine decrescente per creare l'etichetta della classe
        etichetta = "-".join(map(str, sorted(s, reverse=True)))
        
        if etichetta not in classi:
            classi[etichetta] = []
        classi[etichetta].append(s)
        
    return classi

In [ ]:
import csv
import numpy as np

def carica_storico_da_csv(percorso_csv, strategie_giocatore_2):
    """
    Genera il vettore 'storico' (indici delle mosse del giocatore 2)
    a partire da un file CSV salvato dal gioco.

    Parametri
    ---------
    percorso_csv : str
        Percorso del file CSV (es. 'blotto_1791232163824.csv')
    strategie_giocatore_2 : list
        Lista delle strategie generate con genera_strategie(m, l),
        serve per mappare ogni mossa al suo indice.

    Returns
    -------
    storico : np.ndarray di int
        Vettore degli indici, compatibile con cross_validation_2d
        e con i modelli kernel.
    """
    # Dizionario per la conversione mossa -> indice
    mossa_to_index = {mossa: i for i, mossa in enumerate(strategie_giocatore_2)}

    with open(percorso_csv, newline='', encoding='utf-8') as f:
        righe = list(csv.reader(f))

    # Trova la riga di intestazione (quella che inizia con "Round")
    idx_header = next(i for i, r in enumerate(righe) if r and r[0].strip() == 'Round')

    storico = []
    for r in righe[idx_header + 1:]:
        # Salta righe vuote o non numeriche
        if not r or not r[0].strip().isdigit():
            continue
        # Colonne 4,5,6 = giocatore 2 (Collina, Ponte, Bosco)
        mossa = (int(r[4]), int(r[5]), int(r[6]))
        storico.append(mossa_to_index[mossa])

    return np.array(storico, dtype=int)